# MEDDIAG — fresh training run on Kaggle T4 (6 GB VRAM cap, runs with your laptop OFF)

Trains the full pipeline **from scratch**: FAISS index → Stage-1 projector → Stage-2 LoRA + cls_head → eval → 9 experiments → plots.

### Why your laptop can be off
Run this notebook with **Save Version → "Save & Run All (Commit)"**, not by running cells interactively.
A commit run executes on Kaggle's servers with no browser attached. You can close the tab and shut the laptop down; it keeps going for up to 12 h.
Interactive sessions are different: they stop when the tab closes or sits idle.

### What happens each run
1. It downloads the **latest** version of `STATE_DATASET` through the Kaggle API, so you never need to re-attach or update a dataset version. On the very first run the dataset doesn't exist yet, so it starts fresh.
2. It runs `run_pipeline.sh --resume`, which continues exactly where the last run stopped.
3. While training, it uploads checkpoints to `STATE_DATASET` every `UPLOAD_EVERY_MIN` minutes.
4. At `TIME_BUDGET_H` (11 h), it stops training cleanly, uploads one last time, and exits *before* Kaggle's 12 h kill.

**To continue training:** click **Save Version → Save & Run All** again (works from a phone). Repeat until the log prints `ALL STAGES COMPLETE`.

### One-time setup (notebook settings panel)
- **Accelerator:** GPU T4 (T4 x2 is fine; only GPU 0 is used). **Internet:** ON. **Persistence:** none needed.
- **Add-ons → Secrets** (tick each for this notebook): `HF_TOKEN`, `KAGGLE_KEY` (a `KGAT_…` API token or legacy key), optional `SEMANTIC_SCHOLAR_API_KEY`.
- **Add Input** (strongly recommended, avoids HF Xet 403s):
  - Kaggle Model **Llama 3.2 → 3B-Instruct (transformers)**
  - `raddar/chest-xrays-indiana-university` (IU-Xray)
  - `paultimothymooney/chest-xray-pneumonia` (Kermany)
  - *(optional)* any old dataset containing `faiss_index/`. The retrieval index is **text-only** (MiniLM over reports), not a trained model, so reusing it saves the ~1 h index build without affecting the from-scratch training. Nothing else is taken from old datasets.

In [ ]:
# 1. CONFIG ====================================================================
import os, time
RUN_START = time.time()                  # the time budget is measured from here
os.environ["HF_HUB_DISABLE_XET"] = "1"   # HF Xet CDN 403s from Kaggle -> classic LFS (must precede any HF import)

# --- Where checkpoints live between runs --------------------------------------
KAGGLE_USERNAME = "sujalprasad0206"      # account that owns STATE_DATASET (must match the KAGGLE_KEY secret)
STATE_DATASET   = "meddiag-fresh-t4"     # NEW dataset -> clean from-scratch run; created automatically on first upload
FORCE_FRESH     = False                  # True = ignore STATE_DATASET and restart from step 0 (it will be OVERWRITTEN)

# --- Hard constraints -----------------------------------------------------------
VRAM_CAP_GB      = 6.0                   # project constraint: every process is capped to this much VRAM (T4 has 16)
TIME_BUDGET_H    = 11.0                  # stop training at this point; leaves ~1 h for the final upload before Kaggle's 12 h kill
UPLOAD_EVERY_MIN = 20                    # background checkpoint upload cadence

# --- Training knobs (same recipe as the last DenseNet run) ---------------------
VISION_AUGMENT  = True                   # train-time CXR augmentation (OOD robustness)
VISION_FINETUNE = False                  # keep DenseNet-121 frozen (preserves its multi-dataset CXR features)
CLS_POS_WEIGHT  = 2.5                    # up-weight ABNORMAL (recall)
CLS_FOCAL_GAMMA = 2.0                    # focal loss on hard / missed abnormals
CLS_POOL        = "attn"                 # cls_head visual pooling: "attn" | "mean"
MAX_PAIRS_S2    = 4000
GRAD_ACCUM      = 4                      # batch size is 1; accumulation keeps activations small (6 GB-safe)

REPO_URL    = "https://github.com/Sreenjoyee/visual-language-model-research-qlora-cot-rag.git"
REPO_BRANCH = "main"
REPO_DIR    = "/kaggle/working/vlm"

print(f"state dataset : {KAGGLE_USERNAME}/{STATE_DATASET}  (force_fresh={FORCE_FRESH})")
print(f"VRAM cap      : {VRAM_CAP_GB} GB   time budget: {TIME_BUDGET_H} h   upload every {UPLOAD_EVERY_MIN} min")
print(f"recipe        : augment={VISION_AUGMENT} finetune={VISION_FINETUNE} pos_w={CLS_POS_WEIGHT} "
      f"focal={CLS_FOCAL_GAMMA} pool={CLS_POOL} pairs_s2={MAX_PAIRS_S2} accum={GRAD_ACCUM}")

In [ ]:
# 2. PACKAGES ==================================================================
import subprocess, sys
def pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])
pip("--upgrade", "kaggle")
pip("peft>=0.19.1", "bitsandbytes>=0.49.2", "accelerate>=1.13.0", "faiss-cpu==1.13.2",
    "sentence-transformers", "bert-score>=0.3.13", "torchxrayvision>=1.2.0", "python-dotenv")
print("Packages ready.")

In [ ]:
# 3. GPU, SECRETS & ENVIRONMENT ================================================
import os, json, glob
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # T4 x2 -> use one GPU only (6 GB budget is per-GPU)
import torch
from kaggle_secrets import UserSecretsClient

assert torch.cuda.is_available(), "No GPU. Settings -> Accelerator -> GPU T4, then Save & Run All again."
_p = torch.cuda.get_device_properties(0)
print(f"GPU: {_p.name}  {_p.total_memory / 2**30:.1f} GiB total  -> capped to {VRAM_CAP_GB} GB per process")

secrets = UserSecretsClient()
def _secret(name, required=True):
    try:
        return secrets.get_secret(name).strip()
    except Exception as e:
        if required:
            raise RuntimeError(f"Kaggle secret '{name}' is missing. Add-ons -> Secrets -> add it AND tick it for this notebook.") from e
        return ""

os.environ["HF_TOKEN"] = _secret("HF_TOKEN")
_s2 = _secret("SEMANTIC_SCHOLAR_API_KEY", required=False)
if _s2: os.environ["SEMANTIC_SCHOLAR_API_KEY"] = _s2

# Kaggle API auth (used to download / upload STATE_DATASET)
_kkey = _secret("KAGGLE_KEY")
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
_kd = Path.home() / ".kaggle"; _kd.mkdir(exist_ok=True)
if _kkey.startswith("KGAT_"):
    os.environ["KAGGLE_API_TOKEN"] = _kkey
    (_kd / "access_token").write_text(_kkey); (_kd / "access_token").chmod(0o600)
else:
    os.environ["KAGGLE_KEY"] = _kkey
    (_kd / "kaggle.json").write_text(json.dumps({"username": KAGGLE_USERNAME, "key": _kkey}))
    (_kd / "kaggle.json").chmod(0o600)

# 6 GB constraint. MAX_VRAM_GB = where the LLM weights may be placed; VRAM_CAP_GB =
# hard allocator ceiling in EVERY child process (applied by src/config.py on import).
os.environ["MEDDIAG_MAX_VRAM_GB"]     = str(VRAM_CAP_GB)
os.environ["MEDDIAG_VRAM_CAP_GB"]     = str(VRAM_CAP_GB)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less fragmentation under a tight cap
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONUNBUFFERED"]        = "1"
os.environ["MEDDIAG_MIMIC_REPO"]      = "power2004/mimic-cxr-dataset"

os.environ["MEDDIAG_VISION_AUGMENT"]  = "1" if VISION_AUGMENT else "0"
os.environ["MEDDIAG_VISION_FINETUNE"] = "1" if VISION_FINETUNE else "0"
os.environ["MEDDIAG_CLS_POS_WEIGHT"]  = str(CLS_POS_WEIGHT)
os.environ["MEDDIAG_CLS_FOCAL_GAMMA"] = str(CLS_FOCAL_GAMMA)
os.environ["MEDDIAG_CLS_POOL"]        = CLS_POOL

# Prefer an attached Kaggle copy of LLaMA-3.2-3B over the HF download (HF Xet 403s on Kaggle).
def _find_local_llm():
    cands = []
    for cfg in glob.glob("/kaggle/input/**/config.json", recursive=True):
        d = os.path.dirname(cfg)
        if glob.glob(os.path.join(d, "*.safetensors")) and glob.glob(os.path.join(d, "tokenizer*")):
            cands.append(d)
    for d in cands:
        if "llama" in d.lower() and ("3.2" in d or "3b" in d.lower()):
            return d
    return cands[0] if cands else None
_llm = _find_local_llm()
if _llm:
    os.environ["MEDDIAG_LLM_MODEL"] = _llm
    print(f"LLM: local {_llm}")
else:
    print("LLM: HuggingFace download (attach the Kaggle Llama-3.2-3B-Instruct model if this 403s)")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Secrets + env ready.")

In [ ]:
# 4. CLONE REPO ================================================================
import subprocess, os
from pathlib import Path
if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth=1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR])
else:   # interactive re-run: update code, keep untracked models/ logs/
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "--depth=1", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"])
os.chdir(REPO_DIR)
print("commit:", subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())

# The 6 GB hard cap lives in src/config.py. Refuse to train on an old checkout without it.
if "MEDDIAG_VRAM_CAP_GB" not in Path("src/config.py").read_text():
    raise RuntimeError("This repo checkout has no MEDDIAG_VRAM_CAP_GB support in src/config.py. "
                       "Push the latest commit to GitHub, then Save & Run All again.")

In [ ]:
# 5. RESTORE STATE FROM THE LATEST DATASET VERSION (auto-resume) ===============
# The dataset is the single source of truth. It's downloaded via the API (always
# the LATEST version), so an attached-but-stale input version can't cause a resume
# from an old step. First run / FORCE_FRESH -> nothing restored -> from scratch.
import os, re, json, shutil, zipfile, subprocess
from pathlib import Path
import torch

MODELS = Path(REPO_DIR) / "models"
LOGS   = Path(REPO_DIR) / "logs"
FAISS  = Path(REPO_DIR) / "faiss_index"
DL     = Path("/kaggle/working/_restore")

# Local workdir starts clean (commit runs are empty anyway; this also protects interactive re-runs).
for p in (MODELS, DL):
    if p.exists(): shutil.rmtree(p)
    p.mkdir(parents=True)
LOGS.mkdir(exist_ok=True)
for f in (LOGS / ".pipeline_state", LOGS / ".pipeline.lock"):
    f.unlink(missing_ok=True)

def _expand_zips(root):
    """--dir-mode zip uploads each top-level folder as <name>.zip -> unpack to <name>/ (flatten a same-name nest)."""
    for z in list(Path(root).rglob("*.zip")):
        dst = z.with_suffix("")
        dst.mkdir(exist_ok=True)
        with zipfile.ZipFile(z) as zf: zf.extractall(dst)
        nested = dst / dst.name
        if nested.is_dir():
            for q in list(nested.iterdir()): shutil.move(str(q), str(dst / q.name))
            nested.rmdir()
        z.unlink()

def _first(root, name, want_dir=False):
    for r, ds, fs in os.walk(root):
        if name in (ds if want_dir else fs): return Path(r) / name
    return None

def _loads(path):
    try:
        torch.load(path, map_location="cpu", weights_only=False); return True
    except Exception as e:
        print(f"  [restore] unreadable, skipped: {path.name} ({type(e).__name__})"); return False

def _restore_faiss_from(root):
    d = _first(root, "faiss_index", want_dir=True)
    src = d if (d and (d / "index.faiss").exists()) else None
    if src is None:
        idx = _first(root, "index.faiss")
        if idx and (idx.parent / "meta.jsonl").exists(): src = idx.parent
    if src is None: return False
    if FAISS.exists(): shutil.rmtree(FAISS)
    FAISS.mkdir(parents=True)
    for fn in ("index.faiss", "meta.jsonl"): shutil.copy2(src / fn, FAISS / fn)
    return True

def _restore_bundle(root):
    got = {}
    pj = _first(root, "projector_stage1.pt")
    if pj and _loads(pj):
        shutil.copy2(pj, MODELS / pj.name); got["projector"] = True
    else:   # Stage-1 still in progress -> newest READABLE mid-checkpoint
        mids = sorted(((int(m.group(1)), Path(r) / f) for r, _, fs in os.walk(root) for f in fs
                       if (m := re.match(r"projector_step(\d+)\.pt$", f))), reverse=True)
        for step, p in mids:
            if _loads(p):
                shutil.copy2(p, MODELS / p.name); got["stage1_mid"] = step; break
    # Stage-2 checkpoints: copy all, then drop a newest one that is corrupt (killed mid-save),
    # because run_pipeline.sh would otherwise restart Stage-2 from step 0.
    s2 = sorted(((int(m.group(1)), Path(r) / d) for r, ds, _ in os.walk(root) for d in ds
                 if (m := re.match(r"lora_step(\d+)$", d))))
    for _, d in s2: shutil.copytree(d, MODELS / d.name, dirs_exist_ok=True)
    for step, _ in reversed(s2):
        ts = MODELS / f"lora_step{step}" / "train_state.pt"
        if ts.exists() and _loads(ts): got["stage2_step"] = step; break
        shutil.rmtree(MODELS / f"lora_step{step}")
    for name in ("cls_head.pt",):
        f = _first(root, name)
        if f: shutil.copy2(f, MODELS / name); got[name] = True
    for name in ("lora_adapter",):       # final Stage-2 adapter (present only once Stage-2 finished)
        d = _first(root, name, want_dir=True)
        if d: shutil.copytree(d, MODELS / name, dirs_exist_ok=True); got[name] = True
    got["faiss"] = _restore_faiss_from(root)
    for name in (".pipeline_state", "stage1.jsonl", "stage2.jsonl", "run.log"):
        f = _first(root, name)
        if f: shutil.copy2(f, LOGS / name)
    for f in Path(root).rglob("eval_report_*.json"): shutil.copy2(f, LOGS / f.name)
    for sub in ("reports", "diagnostics"):
        d = _first(root, sub, want_dir=True)
        if d: shutil.copytree(d, Path(REPO_DIR) / sub, dirs_exist_ok=True)
    return got

def dataset_exists():
    """True/False from the owner's dataset list. A missing dataset can 403 on download,
    so download errors alone can't tell 'first run' from 'API hiccup'."""
    r = subprocess.run(["kaggle", "datasets", "list", "--mine", "--search", STATE_DATASET, "--csv"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Kaggle API not usable (check the KAGGLE_KEY secret / KAGGLE_USERNAME): "
                           f"{((r.stderr or '') + (r.stdout or '')).strip()[-300:]}")
    want = f"{KAGGLE_USERNAME}/{STATE_DATASET}".lower()
    return any(ln.split(",")[0].strip().lower() == want for ln in r.stdout.splitlines())

got, source = {}, "none"
if FORCE_FRESH:
    print("[restore] FORCE_FRESH=True -> starting from scratch (dataset will be overwritten by the next upload)")
elif not dataset_exists():
    print(f"[restore] {KAGGLE_USERNAME}/{STATE_DATASET} does not exist yet -> FIRST RUN, from scratch")
else:
    r = subprocess.run(["kaggle", "datasets", "download", f"{KAGGLE_USERNAME}/{STATE_DATASET}",
                        "--unzip", "-p", str(DL)], capture_output=True, text=True)
    if r.returncode == 0 and any(DL.iterdir()):
        _expand_zips(DL); got = _restore_bundle(DL); source = "latest dataset version (API)"
    else:
        # Dataset exists but download failed: use the attached mount if present (may be an older version)
        mount = next((q for q in Path("/kaggle/input").rglob(STATE_DATASET) if q.is_dir()), None)
        if mount is None:
            raise RuntimeError(f"{STATE_DATASET} exists but could not be downloaded, and it isn't attached either. "
                               f"Refusing to silently restart from scratch. Kaggle CLI said: {msg}")
        print(f"[restore] API download failed ({msg}); using attached mount {mount} (may be an older version)")
        tmp = DL / "mount"; shutil.copytree(mount, tmp); _expand_zips(tmp)
        got = _restore_bundle(tmp); source = f"attached mount {mount}"

if not FAISS.joinpath("index.faiss").exists():
    # Text-only retrieval index: reuse one from ANY attached input if present (not trained weights).
    if _restore_faiss_from("/kaggle/input"):
        print("[restore] faiss_index reused from an attached input (text index, not trained weights)")
    else:
        print("[restore] no faiss_index anywhere -> pipeline step 1 will BUILD it (~1 h, uploaded afterwards)")
shutil.rmtree(DL, ignore_errors=True)

# Arch guard: a restored projector must be DenseNet-121 (1024-dim vision input).
_pj = next(iter(sorted(MODELS.glob("projector_*.pt"))), None)
if _pj is not None:
    _sd = torch.load(_pj, map_location="cpu", weights_only=False)
    _sd = _sd.get("projector", _sd.get("model", _sd)) if isinstance(_sd, dict) else _sd
    _vp = next((v for k, v in _sd.items() if str(k).endswith("vision_proj.weight")), None) if isinstance(_sd, dict) else None
    if _vp is not None and int(_vp.shape[1]) != 1024:
        raise RuntimeError(f"ARCH MISMATCH: {_pj.name} expects {int(_vp.shape[1])}-dim features; DenseNet-121 gives 1024.")

print(f"restore source : {source}")
print(f"restored       : {got or 'nothing (fresh start)'}")
print(f"faiss present  : {FAISS.joinpath('index.faiss').exists()}")

In [ ]:
# 6. SMOKE TEST + 6 GB CAP VERIFICATION (fails in seconds, not hours in) =======
import subprocess, sys, torch
from PIL import Image
from src.config import CONFIG
from src.vision import VisionEncoder
from src.projector import PerceiverResampler

assert CONFIG.vision_model_id.startswith("xrv:") and CONFIG.vision_hidden_dim == 1024, "encoder is not DenseNet-121"
ve = VisionEncoder(CONFIG)
px = ve.preprocess(Image.new("L", (256, 256), 128))
assert tuple(px.shape) == (1, 1, 224, 224), f"bad preprocess shape {tuple(px.shape)}"
tok = ve(px)
assert tok.shape[-1] == 1024, f"feature dim {tok.shape[-1]} != 1024"
proj = PerceiverResampler(vision_dim=CONFIG.vision_hidden_dim, llm_dim=CONFIG.llm_hidden_dim,
                          num_latents=CONFIG.num_visual_tokens,
                          num_heads=CONFIG.projector_num_heads, num_layers=CONFIG.projector_num_layers)
out = proj(tok.float())
assert tuple(out.shape) == (1, CONFIG.num_visual_tokens, CONFIG.llm_hidden_dim), f"projector out {tuple(out.shape)}"
del ve, proj, tok, out; torch.cuda.empty_cache()
print("[ok] DenseNet-121 -> projector shapes consistent")

# Prove the cap is live in a CHILD process (training runs as child processes):
# allocating VRAM_CAP_GB + 1 GB must fail with CUDA OOM.
_probe = (f"import src.config, torch; "
          f"torch.empty(int({VRAM_CAP_GB + 1} * 2**30), dtype=torch.uint8, device='cuda')")
r = subprocess.run([sys.executable, "-c", _probe], cwd=REPO_DIR, capture_output=True, text=True)
if r.returncode == 0 or "OutOfMemory" not in (r.stderr + r.stdout):
    raise RuntimeError(f"6 GB cap NOT enforced in child processes -- refusing to train.\n{r.stderr[-500:]}")
print(f"[ok] hard VRAM cap active: a {VRAM_CAP_GB + 1:.0f} GB allocation is rejected in child processes")

In [ ]:
# 7. PIPELINE STATE + RUN CONFIG ===============================================
# run_pipeline.sh --resume skips every step listed in logs/.pipeline_state.
# Rebuild it from the restored state, but only trust a marker if its artifact exists.
import re
from pathlib import Path
sp = LOGS / ".pipeline_state"
done = set(sp.read_text().split()) if sp.exists() else set()
done.add("step0")                                               # unit tests: not needed on Kaggle
done.discard("step1"); done.discard("step2"); done.discard("step3")
if (FAISS / "index.faiss").exists() and (FAISS / "meta.jsonl").exists(): done.add("step1")
if (MODELS / "projector_stage1.pt").exists():                   done.add("step2")
if (MODELS / "lora_adapter").is_dir() and (MODELS / "cls_head.pt").exists(): done.add("step3")
if "step3" not in done:                                         # eval needs the final adapter
    done -= {"step4", "step5", "step6", "step7"}
order = [f"step{i}" for i in range(8)]
sp.write_text("".join(s + "\n" for s in order if s in done))
print("already done:", [s for s in order if s in done])
print("to run      :", [s for s in order if s not in done] or "NOTHING (pipeline finished)")

sh = Path(REPO_DIR) / "run_pipeline.sh"; t = sh.read_text()
t = re.sub(r"GRAD_ACCUM=8\b", f"GRAD_ACCUM={GRAD_ACCUM}", t)          # non-smoke value only
t = re.sub(r"MAX_PAIRS_S2=4000\b", f"MAX_PAIRS_S2={MAX_PAIRS_S2}", t)
sh.write_text(t)
print(f"run_pipeline.sh: GRAD_ACCUM={GRAD_ACCUM} MAX_PAIRS_S2={MAX_PAIRS_S2}")

In [ ]:
# 8. TRAIN (time-boxed) + BACKGROUND CHECKPOINT UPLOADS ========================
import os, re, json, time, shutil, signal, subprocess, threading
from pathlib import Path

OUT = Path("/kaggle/working/_upload")
_lock = threading.Lock()
_FRESH_S = 90   # skip files written in the last 90 s (may be mid-save)

def _settled(p):
    p = Path(p)
    newest = max([p.stat().st_mtime] + [q.stat().st_mtime for q in p.rglob("*")]) if p.is_dir() else p.stat().st_mtime
    return time.time() - newest > _FRESH_S

def _fingerprint():
    items = []
    for p in list(MODELS.glob("*")) + [FAISS, LOGS / ".pipeline_state", Path(REPO_DIR) / "reports"]:
        if p.exists(): items.append((p.name, int(p.stat().st_mtime)))
    return tuple(sorted(items))

def package():
    if OUT.exists(): shutil.rmtree(OUT)
    OUT.mkdir(parents=True)
    n = 0
    for p in sorted(MODELS.iterdir()):
        try:   # checkpoint rotation may delete a folder while we copy it -> just skip that one
            if p.name == "lora_adapter_swa" or not _settled(p): continue
            (shutil.copytree if p.is_dir() else shutil.copy2)(p, OUT / p.name); n += 1
        except (FileNotFoundError, shutil.Error) as e:
            shutil.rmtree(OUT / p.name, ignore_errors=True)
            print(f"[upload] skipped {p.name} (changed during copy: {type(e).__name__})")
    if FAISS.joinpath("index.faiss").exists(): shutil.copytree(FAISS, OUT / "faiss_index")
    for name in (".pipeline_state", "stage1.jsonl", "stage2.jsonl", "run.log"):
        if (LOGS / name).exists(): shutil.copy2(LOGS / name, OUT / name)
    for f in LOGS.glob("eval_report_*.json"): shutil.copy2(f, OUT / f.name)
    for sub in ("reports", "diagnostics"):
        d = Path(REPO_DIR) / sub
        if d.is_dir() and any(d.iterdir()): shutil.copytree(d, OUT / sub)
    return n

def upload(msg):
    with _lock:
        n = package()
        if n == 0 and not (OUT / "faiss_index").exists():
            print("[upload] nothing to upload yet"); return False
        (OUT / "dataset-metadata.json").write_text(json.dumps(
            {"title": STATE_DATASET, "id": f"{KAGGLE_USERNAME}/{STATE_DATASET}", "licenses": [{"name": "CC0-1.0"}]}))
        for attempt in range(3):
            try:
                exists = dataset_exists()
            except RuntimeError as e:
                print(f"[upload] attempt {attempt + 1}/3: {e}"); time.sleep(60); continue
            cmd = (["kaggle", "datasets", "version", "-p", str(OUT), "-m", msg, "--dir-mode", "zip"] if exists
                   else ["kaggle", "datasets", "create", "-p", str(OUT), "--dir-mode", "zip"])
            r = subprocess.run(cmd, capture_output=True, text=True)
            blob = (r.stdout or "") + (r.stderr or "")
            if r.returncode == 0 and ("error" not in blob.lower() or "being created" in blob.lower()):
                print(f"[upload] OK ({n} model items) -> {KAGGLE_USERNAME}/{STATE_DATASET}: {msg}"); return True
            print(f"[upload] attempt {attempt + 1}/3 failed: {blob.strip()[-300:]}")
            time.sleep(60)
        return False

stop_uploader = threading.Event()
def _uploader():
    last = _fingerprint()
    while not stop_uploader.wait(UPLOAD_EVERY_MIN * 60):
        try:   # never let one bad cycle kill the uploader thread
            fp = _fingerprint()
            if fp != last:
                hours = (time.time() - RUN_START) / 3600
                if upload(f"autosave at {hours:.1f} h"): last = fp
        except Exception as e:
            print(f"[upload] cycle failed, will retry next cycle: {type(e).__name__}: {e}")
threading.Thread(target=_uploader, daemon=True).start()

deadline = RUN_START + TIME_BUDGET_H * 3600
print(f"[train] budget ends in {(deadline - time.time()) / 3600:.2f} h; uploads every {UPLOAD_EVERY_MIN} min\n")
proc = subprocess.Popen(["bash", "run_pipeline.sh", "--resume"], cwd=REPO_DIR, env=os.environ.copy(),
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                        start_new_session=True)   # own process group -> clean kill of all children
def _pump():
    for line in proc.stdout: print(line, end="", flush=True)
threading.Thread(target=_pump, daemon=True).start()

timed_out = False
while proc.poll() is None:
    if time.time() > deadline:
        timed_out = True
        print(f"\n[train] TIME BUDGET ({TIME_BUDGET_H} h) reached -> stopping; progress since the last checkpoint is discarded")
        os.killpg(proc.pid, signal.SIGTERM)
        try: proc.wait(timeout=120)
        except subprocess.TimeoutExpired: os.killpg(proc.pid, signal.SIGKILL); proc.wait()
        break
    time.sleep(15)
time.sleep(2)
stop_uploader.set()
rc = proc.returncode
print(f"\n[train] run_pipeline.sh exited rc={rc} after {(time.time() - RUN_START) / 3600:.2f} h")

time.sleep(_FRESH_S + 5)          # let the last checkpoint count as settled
final_ok = upload("final upload of this run" + (" (time budget)" if timed_out else ""))

state = (LOGS / ".pipeline_state").read_text().split() if (LOGS / ".pipeline_state").exists() else []
if "step7" in state:
    print("\n==================  ALL STAGES COMPLETE  ==================")
    print("Training, eval, experiments and plots are done. Don't relaunch.")
elif timed_out:
    print("\n>>> NOT FINISHED YET. Click 'Save Version -> Save & Run All' again to continue from the last checkpoint.")
elif rc != 0:
    raise RuntimeError(f"run_pipeline.sh failed (rc={rc}); see the log above. Checkpoints were uploaded "
                       f"(final_ok={final_ok}), so fix the cause and Save & Run All again to resume.")
if not final_ok:
    raise RuntimeError("Final upload FAILED -- this run's progress is only in the notebook Output tab. "
                       "Create a dataset version from the output manually before relaunching.")
print(f"Checkpoints: https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{STATE_DATASET}")

## Reference

**Monitor a run with the laptop off:** kaggle.com on any device → Your Work → this notebook → *Versions* → the running version shows the live log. The `stage1.jsonl` / `stage2.jsonl` logs include `vram_gb` (peak) per step. It must stay under 6.

**Weekly quota:** Kaggle gives about 30 GPU-hours per week, so plan on 2–3 runs a week. Each run picks up where the last one ended.

**Download the weights** (when finished): dataset page → ⋮ → Download, or
```
kaggle datasets download sujalprasad0206/meddiag-fresh-t4 --unzip -p ./out
```

**If something goes wrong**
- `6 GB cap NOT enforced`: the GitHub repo is missing the `MEDDIAG_VRAM_CAP_GB` change in `src/config.py`. Push it.
- CUDA OOM during training: the model genuinely exceeds 6 GB at that step. Lower `GRAD_ACCUM` doesn't help (batch is 1). Check which stage OOMs before changing anything.
- Start over completely: set `FORCE_FRESH = True` for one run, then set it back to `False`.